# TinyCeNN-LM — Global Memory Tournament

**Goal:** close or reverse the remaining NLL gap to the original Transformer attention layer.

This tournament keeps the proven `cellular_adaptive_maxpool5` local/multiscale path and tests global causal memory ideas inspired by:

- **Hedgehog** — learned positive feature maps that mimic softmax attention.
- **Kimi Delta Attention (KDA)** — fine-grained channel-wise forgetting in delta-rule memory.
- **Gated DeltaNet-2** — separates erase and write control in recurrent fast-weight memory.
- **xLSTM/mLSTM** — normalized matrix memory with gated covariance-style updates.
- **Differential Transformer** — subtract a second attention map to suppress irrelevant context.
- **Memory Fusion** — token-wise mixture of Cellular + Hedgehog + GDN2-inspired memory.

The implementations here are intentionally small research approximations for ablation, **not exact reproductions of the authors' optimized kernels**.

**Winner rule:** architecture/rank is selected only by validation NLL. Held-out test data is used afterward. A strict Transformer win requires the paired 95% CI of `delta_nll` to be entirely below zero.


In [ ]:
# Fresh checkout and dependencies
!rm -rf /content/TinyCeNN-LM
!git clone --depth 1 https://github.com/vtavakkoli/TinyCeNN-LM.git /content/TinyCeNN-LM
%cd /content/TinyCeNN-LM
!pip -q install -e .
!pip -q install "transformers==4.57.6" "datasets>=3,<5" "huggingface_hub>=0.34,<2" pandas matplotlib pytest


In [ ]:
import torch, platform
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime required. In Colab choose Runtime > Change runtime type > T4 GPU.")
print("GPU:", torch.cuda.get_device_name(0))


## Choose the search budget

`balanced` reproduces the same data/step budget used by the previous tournament and is the best first comparison.

`quality` spends more optimization steps and evaluates 1024-token extrapolation too.


In [ ]:
PROFILE = "balanced"   # "balanced" or "quality"
SEED = 2026
LAYER = "18"
VARIANTS = ",".join([
    "cellular_adaptive_maxpool5",
    "cellular_hedgehog_global",
    "cellular_kda_global",
    "cellular_gdn2_global",
    "cellular_xlstm_global",
    "cellular_diff_hedgehog",
    "cellular_memory_fusion",
])

PROFILES = {
    "balanced": dict(
        context=256,
        test_contexts="256,512",
        train_documents=64,
        validation_documents=12,
        test_documents=24,
        steps=300,
        lm_steps=40,
        eval_every=50,
        feature_dims="32",
        memory_ranks="16,32",
    ),
    "quality": dict(
        context=512,
        test_contexts="512,1024",
        train_documents=96,
        validation_documents=20,
        test_documents=40,
        steps=600,
        lm_steps=80,
        eval_every=100,
        feature_dims="32",
        memory_ranks="16,32,48",
    ),
}
cfg = PROFILES[PROFILE]
print(PROFILE, cfg)


In [ ]:
# Causality, finite-gradient, checkpoint-reconstruction and sparse-budget checks
!pytest -q tests/test_memory_attention.py tests/test_cellular_attention.py


In [ ]:
from datetime import datetime, timezone
import subprocess, sys, pathlib, json

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = pathlib.Path(f"/content/TinyCeNN-global-memory-results/{PROFILE}-{run_id}")

cmd = [
    sys.executable, "-u", "-m", "scripts.benchmark_global_memory",
    "--layers", LAYER,
    "--variants", VARIANTS,
    "--feature-dims", cfg["feature_dims"],
    "--memory-ranks", cfg["memory_ranks"],
    "--context", str(cfg["context"]),
    "--test-contexts", cfg["test_contexts"],
    "--train-documents", str(cfg["train_documents"]),
    "--validation-documents", str(cfg["validation_documents"]),
    "--test-documents", str(cfg["test_documents"]),
    "--steps", str(cfg["steps"]),
    "--lm-steps", str(cfg["lm_steps"]),
    "--eval-every", str(cfg["eval_every"]),
    "--seed", str(SEED),
    "--output-dir", str(OUT),
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Results:", OUT)


In [ ]:
import pandas as pd, json, numpy as np
from IPython.display import display

val = pd.read_csv(OUT / "validation_summary.csv").sort_values("validation_nll")
display(val[[
    "variant","memory_rank","validation_nll","validation_delta_nll",
    "validation_output_nmse","validation_output_cosine",
    "trainable_parameters","training_seconds"
]].reset_index(drop=True))

selection = json.loads((OUT / "selection.json").read_text())
print("VALIDATION WINNER:", selection["winners"])


In [ ]:
test = pd.read_csv(OUT / "global_memory_summary.csv")
winner_name = next(iter(selection["winners"].values()))
selected = test[(test["candidate"] == winner_name) | (test["candidate"] == "transformer_original")].copy()
cols = [
    "candidate","context","test_nll","test_perplexity","delta_nll",
    "delta_nll_ci_low","delta_nll_ci_high","ppl_ratio",
    "quality","strict_quality_win"
]
display(selected[[c for c in cols if c in selected.columns]].sort_values(["context","candidate"]))

winner_rows = test[test["candidate"] == winner_name].sort_values("context")
for _, r in winner_rows.iterrows():
    if bool(r["strict_quality_win"]):
        verdict = "✅ STRICT HELD-OUT WIN OVER TRANSFORMER"
    elif r["delta_nll"] < 0:
        verdict = "🟡 lower mean NLL, but CI does not establish a strict win"
    elif r["delta_nll_ci_low"] <= 0 <= r["delta_nll_ci_high"]:
        verdict = "🟡 statistically inconclusive vs Transformer"
    else:
        verdict = "❌ Transformer still better"
    print(
        f'context={int(r["context"])}  ΔNLL={r["delta_nll"]:+.6f}  '
        f'95% CI=[{r["delta_nll_ci_low"]:+.6f}, {r["delta_nll_ci_high"]:+.6f}]  {verdict}'
    )


In [ ]:
import matplotlib.pyplot as plt

plot = val.copy()
labels = plot["variant"].str.replace("cellular_","",regex=False) + "_r" + plot["memory_rank"].astype(str)
plt.figure(figsize=(12,5))
plt.bar(range(len(plot)), plot["validation_delta_nll"])
plt.axhline(0, linewidth=1)
plt.axhline(0.02, linewidth=1, linestyle="--")
plt.xticks(range(len(plot)), labels, rotation=70, ha="right")
plt.ylabel("Validation ΔNLL vs Transformer (lower is better)")
plt.title("Global-memory architecture tournament")
plt.tight_layout()
plt.show()


In [ ]:
# Test ΔNLL by context for every candidate
candidates = test[test["candidate"] != "transformer_original"].copy()
plt.figure(figsize=(11,6))
for name, group in candidates.groupby("variant"):
    best_rank = val[val["variant"] == name].iloc[0]["memory_rank"]
    g = group[group["memory_rank"] == best_rank].sort_values("context")
    plt.plot(g["context"], g["delta_nll"], marker="o", label=f"{name} r{int(best_rank)}")
plt.axhline(0, linewidth=1)
plt.axhline(0.02, linewidth=1, linestyle="--")
plt.xlabel("Context")
plt.ylabel("Held-out ΔNLL vs Transformer")
plt.title("Does global memory close the Transformer gap?")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Package all results for review
import shutil, os
archive = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Archive:", archive)
from google.colab import files
files.download(archive)
